# Customizing Aggregations in magg

The aggregation pipeline is driven by a single YAML config file with three
required sections, plus optional fields:

1. **`data_source`** -- what to read: reader, HDF5 groups, coordinates, variables, quality filter
2. **`aggregation`** -- what to compute: coordinate columns plus statistical variables, each wired to a numpy function or a Python expression
3. **`output`** -- where to write: grid spec (type, indexing scheme, child order) and optional store path

Optional top-level fields:
- **`catalog`** -- path to a granule catalog JSON
- **`bounds`** -- temporal/spatial bounds for filtering

Everything -- function dispatch, Zarr template generation, validation --
derives from this config. There is no separate function registry or
hard-coded schema class.

This notebook walks through the default config, then shows two customization
examples:
- Replacing weighted mean with median and dropping quantile fields
- Adding a new input variable (along-track slope) with its own statistics

## Part 1: The current setup

### Loading the default config

`default_config()` loads the built-in `atl06.yaml` shipped with the package
and returns a `PipelineConfig` dataclass with three dict fields.

In [1]:
from magg.config import default_config

cfg = default_config()
print(type(cfg))
print(f"Sections: data_source, aggregation, output")

<class 'magg.config.PipelineConfig'>
Sections: data_source, aggregation, output


### The three sections

Each section is a plain dict parsed from YAML. Let's inspect them.

In [2]:
import json

print("=== data_source ===")
print(json.dumps(cfg.data_source, indent=2))

=== data_source ===
{
  "reader": "h5coro",
  "groups": [
    "gt1l",
    "gt1r",
    "gt2l",
    "gt2r",
    "gt3l",
    "gt3r"
  ],
  "coordinates": {
    "latitude": "/{group}/land_ice_segments/latitude",
    "longitude": "/{group}/land_ice_segments/longitude"
  },
  "variables": {
    "h_li": "/{group}/land_ice_segments/h_li",
    "s_li": "/{group}/land_ice_segments/h_li_sigma"
  },
  "quality_filter": {
    "dataset": "/{group}/land_ice_segments/atl06_quality_summary",
    "value": 0
  }
}


In [3]:
print("=== aggregation ===")
print(json.dumps(cfg.aggregation, indent=2))

=== aggregation ===
{
  "coordinates": {
    "cell_ids": {
      "dtype": "uint64",
      "fill_value": 0
    },
    "morton": {
      "dtype": "int64",
      "fill_value": 0
    }
  },
  "variables": {
    "count": {
      "function": "len",
      "source": "h_li",
      "dtype": "int32",
      "fill_value": 0
    },
    "h_min": {
      "function": "min",
      "source": "h_li",
      "dtype": "float32"
    },
    "h_max": {
      "function": "max",
      "source": "h_li",
      "dtype": "float32"
    },
    "h_mean": {
      "function": "average",
      "source": "h_li",
      "params": {
        "weights": "1.0 / s_li**2"
      },
      "dtype": "float32"
    },
    "h_sigma": {
      "expression": "1.0 / np.sqrt(np.sum(1.0 / s_li**2))",
      "dtype": "float32"
    },
    "h_variance": {
      "function": "var",
      "source": "h_li",
      "dtype": "float32"
    },
    "h_q25": {
      "function": "quantile",
      "source": "h_li",
      "params": {
        "q": 0.25
      },
 

In [4]:
print("=== output ===")
print(json.dumps(cfg.output, indent=2))

=== output ===
{
  "grid": "healpix",
  "indexing_scheme": "nested"
}


### The YAML source

The config above comes from `src/magg/configs/atl06.yaml`. Here is its
content verbatim:

In [5]:
from importlib import resources
import magg.configs

yaml_text = resources.files(magg.configs).joinpath("atl06.yaml").read_text()
print(yaml_text)

data_source:
  reader: h5coro
  groups: [gt1l, gt1r, gt2l, gt2r, gt3l, gt3r]
  coordinates:
    latitude: "/{group}/land_ice_segments/latitude"
    longitude: "/{group}/land_ice_segments/longitude"
  variables:
    h_li: "/{group}/land_ice_segments/h_li"
    s_li: "/{group}/land_ice_segments/h_li_sigma"
  quality_filter:
    dataset: "/{group}/land_ice_segments/atl06_quality_summary"
    value: 0

aggregation:
  coordinates:
    cell_ids:
      dtype: uint64
      fill_value: 0
    morton:
      dtype: int64
      fill_value: 0
  variables:
    count:
      function: len
      source: h_li
      dtype: int32
      fill_value: 0
    h_min:
      function: min
      source: h_li
      dtype: float32
    h_max:
      function: max
      source: h_li
      dtype: float32
    h_mean:
      function: average
      source: h_li
      params:
        weights: "1.0 / s_li**2"
      dtype: float32
    h_sigma:
      expression: "1.0 / np.sqrt(np.sum(1.0 / s_li**2))"
      dtype: float32
    h_va

### Function resolution

`resolve_function()` maps a string name to a callable. The rules are:
- `"len"` or `"count"` resolves to the builtin `len`
- A bare name like `"average"` resolves to `np.average`
- A dotted path like `"np.quantile"` also works

In [ ]:
from magg.config import resolve_function
import numpy as np

for name in ["len", "min", "max", "average", "var", "quantile", "np.median"]:
    func = resolve_function(name)
    if hasattr(np, func.__name__):
        qualified = f"np.{func.__name__}"
    else:
        qualified = f"builtins.{func.__name__}"
    print(f"  {name:20s} -> {qualified}")

### Expression evaluation

Some aggregation variables use `expression:` instead of `function:`. These
are evaluated via `evaluate_expression()` in a restricted namespace
containing numpy and the DataFrame columns as arrays.

For example, `h_sigma` in the default config uses:
```
expression: "1.0 / np.sqrt(np.sum(1.0 / s_li**2))"
```

Let's try it with some synthetic data:

In [7]:
import numpy as np
from magg.config import evaluate_expression

columns = {
    "h_li": np.array([120.5, 118.3, 122.1, 119.7, 121.0], dtype=np.float32),
    "s_li": np.array([0.05, 0.10, 0.03, 0.08, 0.06], dtype=np.float32),
}

# The h_sigma expression: inverse-variance combined uncertainty
expr = "1.0 / np.sqrt(np.sum(1.0 / s_li**2))"
result = evaluate_expression(expr, columns)
print(f"h_sigma = {result:.6f}")

# A simpler example: RMS of h_li
expr2 = "np.sqrt(np.mean(h_li**2))"
result2 = evaluate_expression(expr2, columns)
print(f"h_rms   = {result2:.4f}")

h_sigma = 0.022113
h_rms   = 120.3268


### Dispatch via `calculate_cell_statistics()`

`calculate_cell_statistics()` ties it all together. Given a DataFrame of
observations for a single grid cell, it iterates the aggregation variables
from the config, resolves each function (or evaluates the expression), and
returns a dict of results.

In [8]:
import pandas as pd
from magg.processing import calculate_cell_statistics

# Five synthetic ICESat-2 observations: elevation (h_li) and uncertainty (s_li)
df = pd.DataFrame({
    "h_li": np.array([120.5, 118.3, 122.1, 119.7, 121.0], dtype=np.float32),
    "s_li": np.array([0.05, 0.10, 0.03, 0.08, 0.06], dtype=np.float32),
})

stats = calculate_cell_statistics(df)

for name, value in stats.items():
    print(f"  {name:12s} = {value:.4f}")

  count        = 5.0000
  h_min        = 118.3000
  h_max        = 122.1000
  h_mean       = 121.2685
  h_sigma      = 0.0221
  h_variance   = 1.6256
  h_q25        = 119.7000
  h_q50        = 120.5000
  h_q75        = 121.0000


Notice that `h_mean` is *not* the arithmetic mean -- it's the
inverse-variance weighted mean (via `np.average` with
`weights: "1.0 / s_li**2"`). The observation at 122.1 has `s_li=0.03`
(the most precise), so the weighted mean is pulled toward it.

`h_sigma` is computed by expression, not by function dispatch -- it
evaluates `1.0 / np.sqrt(np.sum(1.0 / s_li**2))` directly on the
column arrays.

## Part 2: Modifying aggregations via YAML

Suppose you want a simpler output: median elevation instead of weighted
mean, and no quantile columns. You write a new YAML config (or edit a
copy of `atl06.yaml`) with only the variables you want:

In [ ]:
import yaml
from magg.config import load_config_from_dict, validate_config, get_agg_fields

simple_yaml = """
data_source:
  reader: h5coro
  groups: [gt1l, gt1r, gt2l, gt2r, gt3l, gt3r]
  coordinates:
    latitude: "/{group}/land_ice_segments/latitude"
    longitude: "/{group}/land_ice_segments/longitude"
  variables:
    h_li: "/{group}/land_ice_segments/h_li"
    s_li: "/{group}/land_ice_segments/h_li_sigma"
  quality_filter:
    dataset: "/{group}/land_ice_segments/atl06_quality_summary"
    value: 0

aggregation:
  coordinates:
    cell_ids: {dtype: uint64, fill_value: 0}
    morton:   {dtype: int64,  fill_value: 0}
  variables:
    count:
      function: len
      source: h_li
      dtype: int32
      fill_value: 0
    h_median:
      function: median
      source: h_li
      dtype: float32
    h_min:
      function: min
      source: h_li
      dtype: float32
    h_max:
      function: max
      source: h_li
      dtype: float32
    h_variance:
      function: var
      source: h_li
      dtype: float32

output:
  grid:
    type: healpix
    indexing_scheme: nested
    child_order: 12
"""

cfg_simple = load_config_from_dict(yaml.safe_load(simple_yaml))
validate_config(cfg_simple)
print("Validated successfully.\n")

print("Aggregation variables:")
for name, meta in get_agg_fields(cfg_simple).items():
    func = meta.get("function", meta.get("expression"))
    print(f"  {name:12s}  {func}")

In [10]:
stats_simple = calculate_cell_statistics(df, config=cfg_simple)

print("Simplified statistics (5 variables instead of 9):")
for name, value in stats_simple.items():
    print(f"  {name:12s} = {value:.4f}")

Simplified statistics (5 variables instead of 9):
  count        = 5.0000
  h_median     = 120.5000
  h_min        = 118.3000
  h_max        = 122.1000
  h_variance   = 1.6256


The median (120.5) differs from the inverse-variance weighted mean because
it isn't influenced by uncertainty -- it's just the middle value.

### Zarr template adapts automatically

The Zarr template is also driven by the config, so it adapts to the
modified variable list:

In [11]:
from magg.schema import xdggs_spec

# Default config -> 9 data vars + 2 coords = 11 arrays
spec_default = xdggs_spec(parent_order=6, child_order=12)
print(f"Default template:    {len(spec_default.members)} arrays")

# Simplified config -> 5 data vars + 2 coords = 7 arrays
spec_simple = xdggs_spec(parent_order=6, child_order=12, config=cfg_simple)
print(f"Simplified template: {len(spec_simple.members)} arrays")
print(f"  {sorted(spec_simple.members.keys())}")

Default template:    11 arrays
Simplified template: 7 arrays
  ['cell_ids', 'count', 'h_max', 'h_median', 'h_min', 'h_variance', 'morton']


## Part 3: Adding a new input variable

The examples above all aggregate `h_li` (elevation). Suppose you also
want statistics on along-track surface slope (`dh_fit_dx`), which lives
in the same ATL06 HDF5 group.

This touches two config sections:
- `data_source.variables` — add the new HDF5 path
- `aggregation.variables` — define statistics that reference it

Here's the full YAML with slope added:

In [ ]:
extended_yaml = """
data_source:
  reader: h5coro
  groups: [gt1l, gt1r, gt2l, gt2r, gt3l, gt3r]
  coordinates:
    latitude: "/{group}/land_ice_segments/latitude"
    longitude: "/{group}/land_ice_segments/longitude"
  variables:
    h_li: "/{group}/land_ice_segments/h_li"
    s_li: "/{group}/land_ice_segments/h_li_sigma"
    dh_fit_dx: "/{group}/land_ice_segments/dh_fit_dx"    # <-- new
  quality_filter:
    dataset: "/{group}/land_ice_segments/atl06_quality_summary"
    value: 0

aggregation:
  coordinates:
    cell_ids: {dtype: uint64, fill_value: 0}
    morton:   {dtype: int64,  fill_value: 0}
  variables:
    # --- elevation stats (same as default) ---
    count:      {function: len,      source: h_li, dtype: int32, fill_value: 0}
    h_min:      {function: min,      source: h_li, dtype: float32}
    h_max:      {function: max,      source: h_li, dtype: float32}
    h_mean:     {function: average,  source: h_li, params: {weights: "1.0 / s_li**2"}, dtype: float32}
    h_sigma:    {expression: "1.0 / np.sqrt(np.sum(1.0 / s_li**2))", dtype: float32}
    h_variance: {function: var,      source: h_li, dtype: float32}
    h_q25:      {function: quantile, source: h_li, params: {q: 0.25}, dtype: float32}
    h_q50:      {function: quantile, source: h_li, params: {q: 0.50}, dtype: float32}
    h_q75:      {function: quantile, source: h_li, params: {q: 0.75}, dtype: float32}
    # --- slope stats (new) ---
    slope_mean:     {function: mean, source: dh_fit_dx, dtype: float32}
    slope_min:      {function: min,  source: dh_fit_dx, dtype: float32}
    slope_max:      {function: max,  source: dh_fit_dx, dtype: float32}
    slope_variance: {function: var,  source: dh_fit_dx, dtype: float32}

output:
  grid:
    type: healpix
    indexing_scheme: nested
    child_order: 12
"""

cfg_extended = load_config_from_dict(yaml.safe_load(extended_yaml))
validate_config(cfg_extended)
print("Extended config validates successfully.\n")

print("All aggregation variables:")
for name in get_agg_fields(cfg_extended):
    print(f"  {name}")

What if we forget to add `dh_fit_dx` to `data_source.variables`?
`validate_config()` catches it:

In [ ]:
# Same config but without dh_fit_dx in data_source.variables
broken_yaml = """
data_source:
  reader: h5coro
  groups: [gt1l, gt1r, gt2l, gt2r, gt3l, gt3r]
  coordinates:
    latitude: "/{group}/land_ice_segments/latitude"
    longitude: "/{group}/land_ice_segments/longitude"
  variables:
    h_li: "/{group}/land_ice_segments/h_li"
    s_li: "/{group}/land_ice_segments/h_li_sigma"
    # dh_fit_dx is missing!

aggregation:
  coordinates:
    cell_ids: {dtype: uint64, fill_value: 0}
    morton:   {dtype: int64,  fill_value: 0}
  variables:
    count: {function: len, source: h_li, dtype: int32, fill_value: 0}
    slope_mean: {function: mean, source: dh_fit_dx, dtype: float32}

output:
  grid:
    type: healpix
    indexing_scheme: nested
    child_order: 12
"""

try:
    validate_config(load_config_from_dict(yaml.safe_load(broken_yaml)))
except ValueError as e:
    print(f"Validation error: {e}")

In [14]:
# Run calculate_cell_statistics with the extended config on synthetic data
df_with_slope = pd.DataFrame({
    "h_li":      np.array([120.5, 118.3, 122.1, 119.7, 121.0], dtype=np.float32),
    "s_li":      np.array([0.05,  0.10,  0.03,  0.08,  0.06],  dtype=np.float32),
    "dh_fit_dx": np.array([0.002, -0.001, 0.005, 0.003, -0.002], dtype=np.float32),
})

stats_ext = calculate_cell_statistics(df_with_slope, config=cfg_extended)

print("Elevation statistics:")
for name, value in stats_ext.items():
    if name.startswith("h_") or name == "count":
        print(f"  {name:12s} = {value:.4f}")

print("\nSlope statistics:")
for name, value in stats_ext.items():
    if name.startswith("slope_"):
        print(f"  {name:16s} = {value:.6f}")

Elevation statistics:
  count        = 5.0000
  h_min        = 118.3000
  h_max        = 122.1000
  h_mean       = 121.2685
  h_sigma      = 0.0221
  h_variance   = 1.6256
  h_q25        = 119.7000
  h_q50        = 120.5000
  h_q75        = 121.0000

Slope statistics:
  slope_mean       = 0.001400
  slope_min        = -0.002000
  slope_max        = 0.005000
  slope_variance   = 0.000007


The Zarr template also picks up the new variables:

In [15]:
spec_ext = xdggs_spec(parent_order=6, child_order=12, config=cfg_extended)
print(f"Extended template arrays ({len(spec_ext.members)}):")
for name in sorted(spec_ext.members.keys()):
    member = spec_ext.members[name]
    print(f"  {name:16s}  dtype={member.data_type}  fill={member.fill_value}")

Extended template arrays (15):
  cell_ids          dtype=uint64  fill=0
  count             dtype=int32  fill=0
  h_max             dtype=float32  fill=NaN
  h_mean            dtype=float32  fill=NaN
  h_min             dtype=float32  fill=NaN
  h_q25             dtype=float32  fill=NaN
  h_q50             dtype=float32  fill=NaN
  h_q75             dtype=float32  fill=NaN
  h_sigma           dtype=float32  fill=NaN
  h_variance        dtype=float32  fill=NaN
  morton            dtype=int64  fill=0
  slope_max         dtype=float32  fill=NaN
  slope_mean        dtype=float32  fill=NaN
  slope_min         dtype=float32  fill=NaN
  slope_variance    dtype=float32  fill=NaN


### Summary

All customization flows through the YAML config (or its dict equivalent):

| What to change | Config section | Key fields |
|----------------|----------------|------------|
| New HDF5 input variable | `data_source.variables` | `column_name: "/{group}/path"` |
| New function-based statistic | `aggregation.variables` | `function:`, `source:`, `params:` |
| New expression-based statistic | `aggregation.variables` | `expression:`, `dtype:` |
| Output grid settings | `output.grid` | `type:`, `indexing_scheme:`, `child_order:` |
| Output store path | `output.store` | local path or `s3://bucket/prefix` |
| Catalog path | `catalog` | top-level, optional |

Rules to remember:
- `function:` and `expression:` are **mutually exclusive** per variable
- Bare function names (e.g. `"min"`) resolve to `np.min`; dotted paths
  like `"np.median"` also work
- `params:` values can be numeric literals, bare column names, or
  expressions containing column names (e.g. `"1.0 / s_li**2"`)
- `validate_config()` checks that all `source:` and param references
  point to columns defined in `data_source.variables`
- The Zarr template (`xdggs_spec()`) and `calculate_cell_statistics()`
  both derive their structure from the same config